# Session 2: Intermediate: the optimization toolkit every engine implements

*LLM Inference Engineering*, session 2 of 3. Each technique twice: first a small implementation you can read in a minute, then the real thing in vLLM with a measurement.

## How the series is divided

| Session | The question it answers | Topics |
|---|---|---|
| **1. Basics** | What happens between a prompt and its answer, and why is it slow? | Tokens, prefill and decode, and the KV cache, built from scratch; Memory math: weights, KV cache, what fits on a GPU; The GPU underneath: memory hierarchy, roofline, the decode speed limit; The metrics: TTFT, TPOT, throughput, goodput, cost per token; First steps with vLLM: offline engine, sampling, the OpenAI server, a load test |
| **2. Intermediate** *(this one)* | Which tricks make serving fast, and what does each one actually do? | Continuous batching and PagedAttention: a scheduler and a block allocator in code; Prefix caching and chunked prefill; Kernels: FlashAttention, CUDA graphs, the roofline measured; Quantization: INT8, FP8, INT4, NF4, AWQ, GPTQ, MXFP4, built and compared; Speculative decoding: draft-and-verify from scratch, n-gram, EAGLE-3, MTP; Structured outputs, scoring, embeddings, LoRA; vLLM vs SGLang |
| **3. Advanced** | How do you serve a model bigger than one GPU, reliably, for many users? | How GPUs connect: NVLink, PCIe, InfiniBand, NCCL, measured; Tensor, pipeline, data and expert parallelism in code, and when to use which; Big models: 70B to DeepSeek and Kimi on 8 GPUs, multi-node; Disaggregated prefill and decode, KV offload, TensorRT-LLM and Dynamo; Hosting open-weight models: recipes, tool calling, reasoning, vision, local machines; Production: Docker and Kubernetes, metrics, priorities, the tuning playbook |

| | |
|---|---|
| **Needs** | A laptop for the from-scratch sections (they use Qwen2.5-0.5B on CPU); one NVIDIA GPU (16 GB+) with vLLM for the rest; SGLang in its own environment for the last sections. |
| **Labels** | *(no GPU needed)* runs on a laptop; *(reading)* has nothing to execute; everything else needs an NVIDIA GPU |
| **Run order** | Top to bottom. Engines are started and stopped per section, so you can also jump to one after running the setup cell. |

In [ ]:
%pip install -q vllm openai matplotlib pandas transformers huggingface_hub pyarrow

In [ ]:
# Shared helpers live in infer_utils.py next to this notebook:
# MODEL, MAX_LEN, PORT, BASE, launch, vllm_serve, stop, shutdown, measure_stream, bench_openai, gpu_mem
from infer_utils import *
try:
    from vllm import LLM, SamplingParams
except ImportError:
    print("vLLM is not installed here; the from-scratch and reading sections still run.")
import os, gc, time, json, subprocess
import numpy as np, pandas as pd, requests
from openai import OpenAI, AsyncOpenAI
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)          # the default model's tokenizer, used for token counts
client = OpenAI(base_url=f"{BASE}/v1", api_key="not-needed")
results = {}                                         # benchmark tables, keyed by engine
print(env_report())

In [ ]:
# The small model the from-scratch sections use (runs on CPU too)
import torch, time, math
from transformers import AutoModelForCausalLM, AutoTokenizer
SMALL = "Qwen/Qwen2.5-0.5B-Instruct"
dev = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if dev == "cuda" else torch.float32
tok_s = AutoTokenizer.from_pretrained(SMALL)
small = AutoModelForCausalLM.from_pretrained(SMALL, torch_dtype=dtype).to(dev).eval()

## Contents

1. Continuous batching from scratch: a scheduler simulation *(no GPU needed)*
2. PagedAttention from scratch: a block allocator with prefix sharing *(no GPU needed)*
3. Prefix caching: cold vs warm TTFT
4. Prefix cache control: reset it, isolate tenants, read the hit rate
5. KV cache capacity & engine knobs
6. Chunked prefill, live: one long prompt vs everyone else's tokens
7. A measured roofline: why batching is free until it isn't
8. FlashAttention vs naive attention: memory is the story
9. Quantization from scratch: what every method actually does *(runs on CPU)*
10. Quantization in vLLM: BF16 vs FP8, measured
11. Speculative decoding from scratch *(runs on CPU, faster on GPU)*
12. Speculative decoding in vLLM (n-gram)
13. Structured output on the server (JSON schema)
14. Scoring text instead of generating it: prompt log-probabilities
15. Beam search
16. Structured outputs: five kinds of constraint
17. Quantize your own checkpoint and check what it cost
18. Pooling models: embeddings, reranking and classification
19. The data-scientist workloads: embeddings and zero-shot labelling at scale
20. Multi-LoRA serving: the server form
21. LoRA adapters: make one, serve several, hot-load at runtime
22. `vllm bench serve`: the official load generator
23. Install & launch an SGLang server
24. Same OpenAI API, plus SGLang's native `/generate`
25. RadixAttention: watch `cached_tokens` and TTFT
26. The SGLang frontend: `fork` for parallel sub-calls
27. One harness, any engine
28. Tuning playbook & cheat sheet *(reading)*

## 1. Continuous batching from scratch: a scheduler simulation *(no GPU needed)*

A toy model of a GPU step: its time is a fixed cost (reading the weights once) plus a small cost per token processed. That's a fair approximation of decode, for the reason Session 1's roofline showed. Two schedulers serve the same 64 requests with random output lengths:

- **Static batching:** fill the batch, wait until **every** sequence is done, then admit the next batch.
- **Continuous batching:** after **every** step, drop finished sequences and admit waiting ones into the free slots.

In [ ]:
import random
random.seed(0)
STEP_FIXED_MS, PER_TOKEN_MS, SLOTS = 20.0, 0.3, 16
reqs = [{"id": i, "arrive": i * 15.0, "prompt": random.randint(50, 500), "out": random.choice([8, 16, 32, 64, 256])} for i in range(64)]

def simulate(policy):
    t, waiting, running, done = 0.0, list(reqs), [], {}
    while waiting or running:
        # admit: static only when the batch is empty, continuous whenever a slot is free
        if policy == "continuous" or not running:
            while waiting and len(running) < SLOTS and waiting[0]["arrive"] <= t:
                r = dict(waiting.pop(0)); r["left"] = r["out"]; r["new"] = True; running.append(r)
        if not running:
            t = waiting[0]["arrive"]; continue
        tokens = sum(r["prompt"] if r["new"] else 1 for r in running)      # prefill tokens + 1 per sequence
        t += STEP_FIXED_MS + PER_TOKEN_MS * tokens
        for r in running:
            r["new"] = False; r["left"] -= 1
        if policy == "continuous":                                         # finished sequences leave now
            for r in running:
                if r["left"] == 0: done[r["id"]] = t - r["arrive"]
            running = [r for r in running if r["left"] > 0]
        elif all(r["left"] <= 0 for r in running):                          # static: the batch returns when its
            for r in running: done[r["id"]] = t - r["arrive"]               # longest member finishes; the rest pad
            running = []
    total_tokens = sum(r["out"] for r in reqs)
    lat = sorted(done.values())
    return {"policy": policy, "makespan_s": t / 1e3, "throughput_tok_s": total_tokens / (t / 1e3),
            "p50_latency_s": lat[len(lat)//2] / 1e3, "p99_latency_s": lat[int(len(lat)*.99)] / 1e3}

display(pd.DataFrame([simulate("static"), simulate("continuous")]).round(2))

Static batching loses twice: short requests sit idle in their slot until the longest one finishes (wasted compute), and new arrivals queue behind the whole batch (worse latency). Continuous batching, introduced by Orca (OSDI '22), fixes both by scheduling at the granularity of one step. vLLM's scheduler does this every iteration, with a **token budget** (`--max-num-batched-tokens`) that caps how many prefill tokens join a step so decodes keep flowing: that's chunked prefill, measured live in this session.

## 2. PagedAttention from scratch: a block allocator with prefix sharing *(no GPU needed)*

vLLM stores the KV cache in fixed-size blocks (16 tokens by default) and keeps a **block table** per sequence, like an operating system's page table. Blocks are allocated only when a sequence needs them, and blocks holding an identical prefix are shared, with a reference count. This toy allocator does exactly that.

In [ ]:
import hashlib
BLOCK = 16

class BlockAllocator:
    def __init__(self, num_blocks):
        self.free = list(range(num_blocks)); self.ref = {}; self.by_hash = {}
    def _hash(self, tokens, parent):
        return hashlib.sha256((str(parent) + ",".join(map(str, tokens))).encode()).hexdigest()[:12]
    def allocate(self, tokens):
        table, parent, hits = [], None, 0
        for i in range(0, len(tokens), BLOCK):
            chunk = tokens[i:i + BLOCK]
            h = self._hash(chunk, parent) if len(chunk) == BLOCK else None   # only full blocks are shareable
            if h in self.by_hash:                                          # prefix cache hit: reuse the block
                b = self.by_hash[h]; self.ref[b] += 1; hits += 1
            else:
                b = self.free.pop(0); self.ref[b] = 1
                if h: self.by_hash[h] = b
            table.append(b); parent = h
        return table, hits
    def release(self, table):
        for b in table:
            self.ref[b] -= 1
            if self.ref[b] == 0:
                self.free.append(b)                                         # (a real cache keeps it for reuse, LRU)

alloc = BlockAllocator(num_blocks=64)
system = list(range(1000, 1064))                     # a 64-token shared system prompt = 4 full blocks
for user in range(3):
    seq = system + [user] * 20                       # each user adds 20 tokens of their own
    table, hits = alloc.allocate(seq)
    print(f"user {user}: block table {table}  shared blocks reused: {hits}")
print(f"blocks in use: {64 - len(alloc.free)} of 64")

In [ ]:
# Why paging matters: memory reserved vs used, for 32 requests with a 4K max length
lens = [random.randint(100, 4096) for _ in range(32)]
reserved = 32 * 4096                                 # pre-vLLM: reserve max_len per request
paged = sum(-(-l // BLOCK) * BLOCK for l in lens)    # paged: round each sequence up to a whole block
print(f"actual tokens {sum(lens):,}   reserve-max-len {reserved:,} ({sum(lens)/reserved:.0%} used)   "
      f"paged {paged:,} ({sum(lens)/paged:.1%} used)")

## 3. Prefix caching: cold vs warm TTFT

vLLM hashes KV-cache blocks by their content plus everything before them. A request that starts with an already-seen prefix **skips prefill** for that part. It's on by default.

We build a ~3,000-token system prompt (think: a long policy doc, tool definitions or RAG context) and compare:
- **cold**: first time this prefix is seen
- **warm**: same prefix, *different* question
- **control**: a different prefix, cold again

In [ ]:
# This section talks to a running vLLM server; start one if it isn't up
if not (isinstance(globals().get("server"), subprocess.Popen) and server.poll() is None):
    server = vllm_serve()

In [ ]:
import uuid
def long_system_prompt(tag):
    clauses = [f"Clause {i}: requests of category {i % 7} must be answered within {i % 5 + 1} business days "
               f"and escalated to tier {i % 3 + 1} support if unresolved." for i in range(110)]
    return f"[doc {tag}] You are a support assistant. Company policy:\n" + "\n".join(clauses)

doc_a = long_system_prompt(uuid.uuid4().hex)
doc_b = long_system_prompt(uuid.uuid4().hex)
print("system prompt tokens:", len(tok.encode(doc_a)) if "tok" in globals() else "~3000")

def ttft(system, question):
    return measure_stream([{"role": "system", "content": system}, {"role": "user", "content": question}],
                          max_tokens=1)["ttft_ms"]

cold    = ttft(doc_a, "What does clause 12 say?")
warm    = ttft(doc_a, "Summarize clause 40.")
control = ttft(doc_b, "What does clause 12 say?")
print(f"cold    : {cold:6.1f} ms\nwarm    : {warm:6.1f} ms   ← prefix reused\ncontrol : {control:6.1f} ms")

> **Prompt design rule:** static content first (system prompt, tools, documents), dynamic content last. Putting a timestamp or user ID at the *top* of the prompt silently destroys your cache hit rate.

> **Watch out.** `tok` comes from Part A. If you restarted the kernel, the token count line just prints an estimate.

In [ ]:
stop(server); print("server stopped |", gpu_mem())

## 4. Prefix cache control: reset it, isolate tenants, read the hit rate

Automatic prefix caching (§3) is on by default. Three controls matter in production:

- `llm.reset_prefix_cache()` / `POST /reset_prefix_cache` (dev mode): clear it before a benchmark so runs are comparable.
- `cache_salt` (a request field on the server): requests with different salts never share cached blocks. Use it per tenant when prompts may contain private data, because cache hits are observable through timing.
- The `vllm:prefix_cache_queries` / `vllm:prefix_cache_hits` counters: hit rate = hits ÷ queries. Low hit rate with shared system prompts usually means something dynamic sits near the top of the prompt.

In [ ]:
llm = LLM(model=MODEL, max_model_len=MAX_LEN, gpu_memory_utilization=0.6)
system = "You are a support agent for a GPU cloud. " + "Policy text. " * 400
def hit_rate():
    m = {x.name: getattr(x, "value", 0) for x in llm.get_metrics()}
    return m.get("vllm:prefix_cache_hits", 0) / max(m.get("vllm:prefix_cache_queries", 1), 1)
sp = SamplingParams(max_tokens=8)
for i in range(4):
    llm.chat([{"role": "system", "content": system}, {"role": "user", "content": f"Question {i}"}], sp, use_tqdm=False)
print(f"hit rate after 4 requests sharing a system prompt: {hit_rate():.0%}")
llm.reset_prefix_cache()
llm.chat([{"role": "system", "content": system}, {"role": "user", "content": "Question 99"}], sp, use_tqdm=False)
print(f"hit rate after a reset (cumulative counter, so it drops): {hit_rate():.0%}")
shutdown(llm); del llm

Each section below starts its own engine. If you get out-of-memory errors, **restart the kernel and re-run the setup cell** at the top of this notebook.

## 5. KV cache capacity & engine knobs

| Flag (`LLM(...)` kwarg) | Controls | Trade-off |
|---|---|---|
| `gpu_memory_utilization` | Share of VRAM vLLM may use | More → bigger KV cache → more concurrency |
| `max_model_len` | Max context per request | Lower → more sequences fit |
| `max_num_seqs` | Max sequences per step | Higher → throughput; lower → per-user speed |
| `max_num_batched_tokens` | Token budget per step (chunked prefill) | Smaller → smoother TPOT; larger → faster TTFT |
| `kv_cache_dtype="fp8"` | KV cache precision | ~2× KV capacity, tiny quality cost |
| `enable_prefix_caching` | Prefix reuse (default on) | Free win when prompts share prefixes |

### Experiment: how `gpu_memory_utilization` changes capacity

Watch the `GPU KV cache size` and `Maximum concurrency` log lines for each run.

In [ ]:
from vllm import LLM, SamplingParams

def kv_capacity(**kw):
    llm = LLM(model=MODEL, max_model_len=MAX_LEN, **kw)
    blocks = None
    try:   # not exposed identically in every version; the startup log is the source of truth
        blocks = llm.llm_engine.vllm_config.cache_config.num_gpu_blocks
    except Exception:
        pass
    shutdown(llm); del llm
    return blocks

for util in [0.4, 0.8]:
    b = kv_capacity(gpu_memory_utilization=util)
    print(f"\n>>> gpu_memory_utilization={util}: num_gpu_blocks={b} "
          f"(× 16 tokens/block ≈ {b*16 if b else '?'} tokens)\n")

> Weights are a fixed cost. **Every extra GB you give vLLM goes to KV cache**, i.e. straight into concurrency.

## 6. Chunked prefill, live: one long prompt vs everyone else's tokens

Sixteen users are streaming answers. Halfway through, one request arrives with a ~3,500-token prompt. We record the **longest gap between tokens** the streaming users see (the stall) and the long request's TTFT, for two per-step token budgets.

In [ ]:
async def stall_test(budget):
    srv = launch(["vllm", "serve", MODEL, "--port", str(PORT), "--max-model-len", str(MAX_LEN),
                  "--gpu-memory-utilization", "0.80", "--max-num-batched-tokens", str(budget),
                  "--no-enable-prefix-caching"], "vllm_chunk.log", f"{BASE}/health")
    ac = AsyncOpenAI(base_url=f"{BASE}/v1", api_key="none")
    async def streamer(i):
        gaps, last = [], None
        s = await ac.chat.completions.create(model=MODEL, max_tokens=400, stream=True, extra_body={"ignore_eos": True},
                                             messages=[{"role": "user", "content": f"Count slowly, user {i}."}])
        async for c in s:
            now = time.perf_counter()
            if last: gaps.append(now - last)
            last = now
        return max(gaps)
    async def long_one():
        await asyncio.sleep(1.0)
        t0 = time.perf_counter()
        s = await ac.chat.completions.create(model=MODEL, max_tokens=8, stream=True,
                                             messages=[{"role": "user", "content": "word " * 3500 + "\nSummarise."}])
        async for c in s: return time.perf_counter() - t0
    res = await asyncio.gather(*[streamer(i) for i in range(16)], long_one())
    srv.terminate(); srv.wait(60)
    return {"max_num_batched_tokens": budget, "worst_stall_ms": max(res[:-1]) * 1e3, "long_prompt_ttft_ms": res[-1] * 1e3}

display(pd.DataFrame([await stall_test(512), await stall_test(8192)]).round(0))

> A small budget splits the long prefill across many steps: the streaming users barely notice, but the long request waits longer for its first token. A big budget does the opposite. There is no free setting, only a choice of which user pays. Disaggregation (Session 3 §3) removes the trade-off by giving prefill its own GPUs.

Four experiments that the popular write-ups (kipply, Horace He, the Hugging Face continuous-batching post) describe in prose. Here you measure them.

## 7. A measured roofline: why batching is free until it isn't

A decode step is mostly `activations @ weights`, with the activation matrix having one row per sequence in the batch. Time a single 8192×8192 BF16 weight multiply as the batch grows. While memory-bound, the time stays flat (you're paying to read the weights anyway). Past the ridge point it starts climbing.

In [ ]:
if torch.cuda.is_available():
    dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    Wt = torch.randn(8192, 8192, dtype=dt, device="cuda")
    rows = []
    for B in [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]:
        x = torch.randn(B, 8192, dtype=dt, device="cuda")
        for _ in range(5): x @ Wt
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for _ in range(50): x @ Wt
        torch.cuda.synchronize(); t = (time.perf_counter() - t0) / 50
        rows.append({"batch": B, "time_us": t * 1e6, "TFLOPS": 2 * B * 8192 * 8192 / t / 1e12})
    roof = pd.DataFrame(rows); display(roof.round(1))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
    a1.plot(roof.batch, roof.time_us, "o-"); a1.set(xscale="log", xlabel="batch (rows)", ylabel="µs per matmul", title="Flat = memory-bound: extra rows are free")
    a2.plot(roof.batch, roof.TFLOPS, "o-"); a2.set(xscale="log", yscale="log", xlabel="batch (rows)", ylabel="achieved TFLOPS", title="The roofline, measured")
    plt.tight_layout(); plt.show()
    del Wt, x; torch.cuda.empty_cache()

> The batch size where the time starts rising is your GPU's real ridge point, close to the ops:byte ratio from Session 1 §5. Below it, a decode step serving 64 users costs about the same as one serving 1 user. That single plot is the economic case for continuous batching.

## 8. FlashAttention vs naive attention: memory is the story

Naive attention materialises the full N×N score matrix in HBM. FlashAttention tiles it through on-chip SRAM, so peak memory grows linearly with sequence length instead of quadratically.

In [ ]:
import torch.nn.functional as F
if torch.cuda.is_available():
    dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    def naive(q, k, v):
        s = (q @ k.transpose(-1, -2)) / q.shape[-1] ** 0.5          # [B, H, N, N] lives in HBM
        s = s + torch.triu(torch.full_like(s[0, 0], float("-inf")), 1)
        return torch.softmax(s, -1) @ v
    rows = []
    for N in [1024, 2048, 4096, 8192, 16384]:
        q, k, v = (torch.randn(1, 16, N, 128, dtype=dt, device="cuda") for _ in range(3))
        for name, fn in [("naive", naive), ("fused (SDPA/flash)", lambda q, k, v: F.scaled_dot_product_attention(q, k, v, is_causal=True))]:
            try:
                torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); base = torch.cuda.memory_allocated()
                fn(q, k, v); torch.cuda.synchronize(); t0 = time.perf_counter()
                for _ in range(5): fn(q, k, v)
                torch.cuda.synchronize()
                rows.append({"seq_len": N, "kernel": name, "ms": (time.perf_counter() - t0) / 5 * 1e3,
                             "peak_extra_MB": (torch.cuda.max_memory_allocated() - base) / 1e6})
            except torch.OutOfMemoryError:
                rows.append({"seq_len": N, "kernel": name, "ms": None, "peak_extra_MB": "OOM"})
        del q, k, v
    display(pd.DataFrame(rows).pivot(index="seq_len", columns="kernel").round(1))

> Doubling the sequence length roughly quadruples the naive kernel's extra memory and leaves the fused kernel's nearly flat. Same maths, same result (FlashAttention is exact), very different traffic to HBM.

## 9. Quantization from scratch: what every method actually does *(runs on CPU)*

All weight quantization is the same two lines: pick a **scale** (and maybe a **zero point**), then round.

`q = round(w / scale) + zero`, and at run time `w ≈ (q - zero) × scale`

The methods differ in three choices: **how many bits**, **how many weights share one scale** (granularity), and **how they pick the values** (plain rounding, or something smarter that minimises the error that matters). We build each one on real weights from Qwen2.5-0.5B.

In [ ]:
import torch
torch.manual_seed(0)
W = small.model.layers[10].mlp.down_proj.weight.detach().float().cpu()      # a real weight matrix
print("weight matrix", tuple(W.shape), f"| range {W.min():.3f} .. {W.max():.3f} | std {W.std():.4f}")

def rel_err(a, b):
    return ((a - b).norm() / a.norm()).item()

# 1) Symmetric absmax INT8, one scale for the whole tensor
def int8_per_tensor(w):
    scale = w.abs().max() / 127
    q = torch.round(w / scale).clamp(-127, 127).to(torch.int8)
    return q.float() * scale

# 2) Same, but one scale per output row (per-channel): an outlier only hurts its own row
def int8_per_channel(w):
    scale = w.abs().amax(dim=1, keepdim=True) / 127
    return torch.round(w / scale).clamp(-127, 127) * scale

# 3) Asymmetric (zero point): uses the full range when values aren't centred on zero
def uint8_asym(w):
    lo, hi = w.amin(dim=1, keepdim=True), w.amax(dim=1, keepdim=True)
    scale = (hi - lo) / 255; zero = torch.round(-lo / scale)
    q = (torch.round(w / scale) + zero).clamp(0, 255)
    return (q - zero) * scale

# 4) Group-wise INT4: one scale per 128 weights, what AWQ, GPTQ and GGUF Q4 all use
def int4_group(w, group=128):
    g_ = w.reshape(-1, group)
    scale = g_.abs().amax(dim=1, keepdim=True) / 7
    return (torch.round(g_ / scale).clamp(-8, 7) * scale).reshape(w.shape)

for name, fn, bits in [("INT8 per-tensor", int8_per_tensor, 8), ("INT8 per-channel", int8_per_channel, 8),
                       ("UINT8 asymmetric", uint8_asym, 8), ("INT4 per-tensor", lambda w: int4_group(w, w.numel()), 4),
                       ("INT4 group-128", int4_group, 4)]:
    print(f"{name:18} {bits} bits   relative error {rel_err(W, fn(W)):.4f}")

Read the last two rows: same 4 bits, but group-wise scales cut the error by more than six times. The price is storage for the scales: one FP16 scale per 128 weights adds 16/128 = 0.125 bits, so "4-bit" models really cost about **4.125–4.5 bits per weight** (that's where the deck's 0.53–0.56 bytes per parameter comes from).

In [ ]:
# 5) NF4 (QLoRA): 16 levels placed at quantiles of a normal distribution, because weights are roughly normal
normal = torch.distributions.Normal(0, 1)
qs = torch.linspace(0.5 / 16, 1 - 0.5 / 16, 16)
NF4 = normal.icdf(qs); NF4 = NF4 / NF4.abs().max()                     # 16 levels in [-1, 1], dense near 0

def nf4_group(w, group=64):
    g_ = w.reshape(-1, group); absmax = g_.abs().amax(dim=1, keepdim=True)
    idx = ((g_ / absmax).unsqueeze(-1) - NF4).abs().argmin(-1)          # nearest level
    return (NF4[idx] * absmax).reshape(w.shape)

# 6) FP8 (E4M3): a float format, so precision is relative; scale only to fit the range (max 448)
def fp8_per_tensor(w):
    scale = w.abs().max() / 448
    return (w / scale).to(torch.float8_e4m3fn).float() * scale

print(f"INT4 group-64   relative error {rel_err(W, int4_group(W, 64)):.4f}")
print(f"NF4  group-64   relative error {rel_err(W, nf4_group(W)):.4f}   (same bits, levels shaped like the data)")
print(f"FP8  per-tensor relative error {rel_err(W, fp8_per_tensor(W)):.4f}   (no groups needed: floats handle the range)")
print("levels:", [round(x, 2) for x in NF4.tolist()])

### Smarter than rounding: AWQ and GPTQ

Plain rounding (RTN) treats every weight as equally important. They aren't: what matters is the error in the layer's **output**, `X @ W.T`, on real inputs.

- **AWQ** (activation-aware): a few input channels carry much larger activations. Scale those weight columns **up** before quantizing (so they lose less precision) and fold the inverse scale into the previous layer. No weights are changed in value; the rounding just lands better where it counts.
- **GPTQ**: quantize one column at a time and **push each column's rounding error onto the columns not yet quantized**, weighted by the inverse Hessian `(XᵀX)⁻¹` from calibration data, so later columns compensate.

Both need a small calibration set and give the biggest gains at 4 bits. Here are both on one layer, against RTN, measuring output error on held-out inputs.

In [ ]:
# calibration and test activations: the real inputs to this layer on some text
acts = {}
h = small.model.layers[10].mlp.down_proj.register_forward_hook(lambda m, i, o: acts.setdefault("x", []).append(i[0].detach().float().cpu()))
import pandas as pd
# Real text for calibration and testing: WikiText-2 (the standard quantization benchmark), distinct chunks for each
from huggingface_hub import hf_hub_download
wiki = "\n".join(pd.read_parquet(hf_hub_download("Salesforce/wikitext", "wikitext-2-raw-v1/test-00000-of-00001.parquet",
                                                  repo_type="dataset")).text)
wiki_ids = tok_s(wiki[:400_000], return_tensors="pt").input_ids[0]
chunks = [wiki_ids[i:i + 512] for i in range(0, 512 * 24, 512)]         # 24 chunks of 512 tokens
CAL, TEST = chunks[:16], chunks[16:]                                    # 8,192 calibration tokens, 4,096 test tokens
def collect(chunks_):
    acts.clear()
    with torch.no_grad():
        for c in chunks_: small(c.unsqueeze(0).to(dev))
    return torch.cat([a.reshape(-1, a.shape[-1]) for a in acts["x"]])
Xcal, Xtest = collect(CAL), collect(TEST)
h.remove()
print(f"calibration rows {len(Xcal)}, test rows {len(Xtest)}")
out_err = lambda Wq: ((Xtest @ W.T - Xtest @ Wq.T).norm() / (Xtest @ W.T).norm()).item()

def rtn_int4_rows(w, group=128):
    return int4_group(w, group)

def awq_int4(w, x, group=128):
    sal = x.abs().mean(0).clamp(min=1e-5)                         # per-input-channel activation size
    best = None
    for alpha in torch.linspace(0, 1, 11):                        # AWQ searches the exponent per layer
        s = sal ** alpha; s = s / s.mean()
        wq = int4_group(w * s, group) / s                         # scale up, quantize, scale back (folded in practice)
        e = ((x @ w.T - x @ wq.T).norm()).item()                  # judged on CALIBRATION inputs only
        if best is None or e < best[0]: best = (e, wq, alpha.item())
    print(f"  AWQ picked alpha = {best[2]:.1f}")
    return best[1]

def gptq_int4(w, x, group=128, damp=0.01):
    w = w.clone(); n = w.shape[1]
    H = 2 * x.T @ x; H += damp * H.diag().mean() * torch.eye(n)
    Hinv = torch.linalg.cholesky(torch.cholesky_inverse(torch.linalg.cholesky(H)), upper=True)
    Q = torch.zeros_like(w)
    for i in range(n):
        if i % group == 0:                                        # one scale per row per group, from current weights
            scale = w[:, i:i + group].abs().amax(dim=1) / 7
        q = torch.round(w[:, i] / scale).clamp(-8, 7) * scale
        Q[:, i] = q
        err = (w[:, i] - q) / Hinv[i, i]
        w[:, i + 1:] -= err.unsqueeze(1) * Hinv[i, i + 1:].unsqueeze(0)   # push the error onto later columns
    return Q

for name, Wq in [("RTN  INT4 g128", rtn_int4_rows(W)), ("AWQ  INT4 g128", awq_int4(W, Xcal)), ("GPTQ INT4 g128", gptq_int4(W, Xcal))]:
    print(f"{name}: output error on held-out inputs {out_err(Wq):.4f}")

On a run with 8,192 calibration tokens from WikiText-2, GPTQ cut this layer's output error by about 10% versus rounding and AWQ by about 1%. Small per layer, but a model has ~170 of these layers and the errors compound, which is why published AWQ and GPTQ 4-bit checkpoints stay much closer to the original than plain rounding does. AWQ helps most on layers with a few dominant input channels; GPTQ helps everywhere but needs the Hessian, so it's slower to produce.

### The same methods on a whole model

Quantization error per matrix is a proxy. What you care about is the model. Below, every linear layer in Qwen2.5-0.5B is fake-quantized (quantize, then dequantize back to float) and we measure perplexity on a few sentences. This is the check to run on your own model and data before trusting any 4-bit checkpoint.

In [ ]:
import copy, math
@torch.no_grad()
def ppl(m):                                                   # WikiText-2 perplexity on the 8 held-out chunks
    nll = sum(m(c.unsqueeze(0).to(dev), labels=c.unsqueeze(0).to(dev)).loss.item() for c in TEST)
    return math.exp(nll / len(TEST))

def fake_quant(model, fn):
    m = copy.deepcopy(model)
    for name, mod in m.named_modules():                     # lm_head is tied to the embeddings; leave it in BF16
        if isinstance(mod, torch.nn.Linear) and "lm_head" not in name and mod.weight.shape[1] % 128 == 0:
            mod.weight.data = fn(mod.weight.data.float().cpu()).to(mod.weight.dtype).to(mod.weight.device)
    return m

rows = [{"format": "original", "perplexity": ppl(small)}]
for name, fn in [("INT8 per-channel", int8_per_channel), ("FP8 per-tensor", fp8_per_tensor),
                 ("INT4 group-128 (RTN)", int4_group), ("NF4 group-64", nf4_group), ("INT4 per-tensor", lambda w: int4_group(w, w.numel()))]:
    mq = fake_quant(small, fn); rows.append({"format": name, "perplexity": ppl(mq)}); del mq
display(pd.DataFrame(rows).round(2))

**The map of formats you'll meet:**

| Format | Bits | Granularity | How values are chosen | Where you see it |
|---|---|---|---|---|
| INT8 W8A16 | 8 | per channel | rounding | Older GPUs, simple and safe |
| **FP8 W8A8** | 8 | per tensor or per block of 128 | float cast | Default first step on H100/H200/B200/Ada; vLLM `--quantization fp8` |
| SmoothQuant W8A8 INT8 | 8 | per channel | moves activation outliers into the weights first | INT8 tensor cores (A100) |
| **AWQ** | 4 | group of 128 | activation-aware scaling | `--quantization awq`, Marlin kernels |
| **GPTQ** | 4 (or 3, 8) | group of 128 | error compensation with the Hessian | `--quantization gptq` |
| NF4 (bitsandbytes) | 4 | group of 64 | normal-quantile levels | QLoRA fine-tuning; slow for serving |
| GGUF k-quants (Q4_K_M…) | 2–8 | super-blocks of 256 | mixed per layer | llama.cpp, Ollama, LM Studio |
| **MXFP4 / NVFP4** | 4 | group of 32 / 16 | float levels with a shared scale | Native on Blackwell; gpt-oss ships MXFP4 |
| FP8 KV cache | 8 | per tensor | float cast | `--kv-cache-dtype fp8`: twice the context or users |

Rule of thumb: weight-only 4-bit (AWQ, GPTQ) speeds up **decode at small batch**, because decode is bandwidth-bound and there are 4× fewer bytes to read. W8A8 (FP8) speeds up **prefill and large batches**, because it also uses the faster low-precision tensor cores.

## 10. Quantization in vLLM: BF16 vs FP8, measured

| Variant | What's quantized | Helps |
|---|---|---|
| **AWQ / GPTQ INT4** (W4A16) | weights → 4-bit | memory-bound decode, small batches; 4× smaller weights |
| **FP8** (W8A8) | weights + activations → 8-bit | compute-bound prefill & big batches (Ada/Hopper+) |
| **FP8 KV cache** | KV cache → 8-bit | ~2× more tokens in the cache |

We compare the BF16 model with its official AWQ 4-bit checkpoint on single-stream latency and batch throughput, and eyeball output quality.

In [ ]:
def profile(model, label, **kw):
    llm = LLM(model=model, max_model_len=MAX_LEN, gpu_memory_utilization=0.6, **kw)
    sp = SamplingParams(temperature=0, max_tokens=128, ignore_eos=True)
    llm.generate(["warmup"], sp, use_tqdm=False)
    t0 = time.perf_counter(); llm.generate(["Explain the roofline model."], sp, use_tqdm=False)
    single = 128 / (time.perf_counter() - t0)
    batch = [f"Explain topic {i} in machine learning." for i in range(128)]
    t0 = time.perf_counter(); llm.generate(batch, sp, use_tqdm=False)
    thr = 128 * 128 / (time.perf_counter() - t0)
    sample = llm.generate(["In one sentence, what is a KV cache?"],
                          SamplingParams(temperature=0, max_tokens=60), use_tqdm=False)[0].outputs[0].text
    shutdown(llm); del llm
    return {"variant": label, "single_stream_tok_s": round(single), "batch128_tok_s": round(thr), "sample": sample.strip()}

quant_rows = [profile(MODEL, "BF16"),
              profile("Qwen/Qwen2.5-1.5B-Instruct-AWQ", "AWQ INT4")]
display(pd.DataFrame(quant_rows))

In [ ]:
# FP8 KV cache: same weights, half-size cache. Compare the "GPU KV cache size" log line with BF16's.
# Needs a GPU/backend with FP8 KV support (Ada / Hopper / Blackwell recommended).
try:
    quant_rows.append(profile(MODEL, "BF16 + FP8 KV cache", kv_cache_dtype="fp8"))
    display(pd.DataFrame(quant_rows))
except Exception as e:
    print("FP8 KV cache not supported here:", e)

> On a model this small, weights are already tiny, so INT4 gains are modest and kernel overheads dominate. On 70B-class models the same switch is the difference between "needs 2 GPUs" and "fits on 1".
>
> **Watch out.** Always run **your own task evals** before shipping a quantized model. Perplexity alone hides regressions.

## 11. Speculative decoding from scratch *(runs on CPU, faster on GPU)*

A small **draft** model guesses `k` tokens cheaply, one by one. The big **target** model then checks all `k` in **one** forward pass, the same cost as a single decode step because decode is bandwidth-bound and processing k+1 tokens costs about the same as 1. Accept the longest prefix the target agrees with, plus one token from the target itself, so every round produces at least one token.

With greedy decoding the check is exact: accept a drafted token if it equals the target's argmax at that position. The output is **identical** to running the target alone.

In [ ]:
TARGET = "Qwen/Qwen2.5-1.5B-Instruct"                     # same tokenizer as the 0.5B draft
target = AutoModelForCausalLM.from_pretrained(TARGET, torch_dtype=dtype).to(dev).eval()
draft = small

@torch.no_grad()
def greedy(model, ids, n):
    for _ in range(n):
        nxt = model(ids).logits[:, -1].argmax(-1, keepdim=True)
        ids = torch.cat([ids, nxt], 1)
    return ids

@torch.no_grad()
def speculative(ids, n, k=4):
    start, target_passes, accepted_total = ids.shape[1], 0, 0
    while ids.shape[1] - start < n:
        guess = greedy(draft, ids, k)[:, ids.shape[1]:]                 # 1) draft k tokens
        logits = target(torch.cat([ids, guess], 1)).logits              # 2) ONE target pass over all of them
        target_passes += 1
        want = logits[:, ids.shape[1] - 1:].argmax(-1)                  # target's choice at each position (k+1)
        agree = (want[:, :k] == guess)[0].int()
        a = int(agree.cumprod(0).sum())                                 # length of the agreeing prefix
        accepted_total += a
        ids = torch.cat([ids, guess[:, :a], want[:, a:a + 1]], 1)       # 3) keep it + one target token
    return ids[:, :start + n], target_passes, accepted_total

prompt_ids = tok_s("Write a Python function that returns the sum of a list of numbers.", return_tensors="pt").input_ids.to(dev)
N = 48
t0 = time.perf_counter(); ref = greedy(target, prompt_ids, N); t_plain = time.perf_counter() - t0
t0 = time.perf_counter(); out, passes, acc = speculative(prompt_ids, N, k=4); t_spec = time.perf_counter() - t0
print("identical output:", torch.equal(ref, out))
print(f"target forward passes: {N} plain vs {passes} speculative  ({N/passes:.2f} tokens per target pass)")
print(f"draft acceptance rate: {acc/(passes*4):.0%}   wall time {t_plain:.1f}s -> {t_spec:.1f}s")
print(tok_s.decode(out[0, prompt_ids.shape[1]:]))

### With sampling: the acceptance rule

When sampling with temperature, accept a drafted token `x` with probability `min(1, p(x) / q(x))`, where `p` is the target's distribution and `q` the draft's. On rejection, sample from the leftover `max(0, p − q)`, renormalised. This rule (Leviathan et al., 2023) makes the output distribution **exactly** the target's. Here it is on a toy vocabulary, checked empirically:

In [ ]:
p = torch.tensor([0.50, 0.30, 0.15, 0.05])          # target distribution over 4 tokens
q = torch.tensor([0.25, 0.25, 0.25, 0.25])          # a poor draft

def spec_sample(p, q):
    x = torch.multinomial(q, 1).item()               # draft proposes
    if torch.rand(1).item() < min(1.0, (p[x] / q[x]).item()):
        return x, True                               # accepted
    resid = (p - q).clamp(min=0); resid /= resid.sum()
    return torch.multinomial(resid, 1).item(), False # rejected: resample from what the draft under-weights

draws = [spec_sample(p, q) for _ in range(50_000)]
emp = torch.bincount(torch.tensor([d[0] for d in draws]), minlength=4).float() / len(draws)
print("target p     :", p.tolist())
print("speculative  :", [round(x, 3) for x in emp.tolist()])
print(f"acceptance rate {sum(d[1] for d in draws)/len(draws):.2f}  (theory: sum of min(p, q) = {torch.minimum(p, q).sum():.2f})")

In [ ]:
# When does it pay? Expected tokens per target pass with acceptance rate a and k drafted tokens:
#   E = (1 - a^(k+1)) / (1 - a)
# Speed-up also depends on the draft's cost c (draft step time / target step time): speedup = E / (k*c + 1)
rows = []
for a in (0.5, 0.7, 0.8, 0.9):
    for k in (2, 4, 7):
        E = (1 - a ** (k + 1)) / (1 - a)
        rows.append({"acceptance": a, "k": k, "tokens_per_pass": round(E, 2),
                     "speedup_draft_5pct": round(E / (k * 0.05 + 1), 2), "speedup_draft_20pct": round(E / (k * 0.20 + 1), 2)})
display(pd.DataFrame(rows).pivot(index="acceptance", columns="k", values="speedup_draft_5pct"))

Three conclusions the engines' defaults are built on:

1. **Acceptance rate is everything.** EAGLE-3 heads (trained on the target's own hidden states) reach 0.7–0.9; a generic small model often 0.5. N-gram drafting costs nothing and is great when the output copies the input (code edits, RAG), useless otherwise.
2. **The draft must be cheap.** At 20% of the target's cost, a mediocre draft makes things slower.
3. **It's a low-concurrency trick.** Verification uses the compute decode leaves idle. At high batch sizes there's no idle compute left, so the gain shrinks or turns negative (Session 2 measures this on vLLM).

## 12. Speculative decoding in vLLM (n-gram)

**Idea:** decode leaves compute idle, so guess *k* tokens cheaply and let the big model verify all of them in **one** forward pass. The output distribution is unchanged.

**N-gram / prompt lookup** needs no extra model: it proposes continuations by copying spans that already appear in the prompt. Ideal for **code editing, rewriting, RAG and summarization**, where the output repeats the input.

```python
speculative_config = {"method": "ngram", "num_speculative_tokens": 4,
                      "prompt_lookup_min": 2, "prompt_lookup_max": 4}
```

In [ ]:
CODE = '''
def compute_stats(values):
    total = 0
    count = 0
    for v in values:
        total = total + v
        count = count + 1
    mean = total / count
    variance = 0
    for v in values:
        variance = variance + (v - mean) ** 2
    variance = variance / count
    return {"mean": mean, "variance": variance, "count": count}
'''
edit_prompt = [{"role": "user", "content":
    "Rename the variable `total` to `running_sum` in this code. Return the full code only.\n" + CODE}]

def time_single(llm, runs=3):
    sp = SamplingParams(temperature=0, max_tokens=256)
    llm.chat(edit_prompt, sp, use_tqdm=False)             # warmup
    ts, n = [], 0
    for _ in range(runs):
        t0 = time.perf_counter(); o = llm.chat(edit_prompt, sp, use_tqdm=False)[0].outputs[0]
        ts.append(time.perf_counter() - t0); n = len(o.token_ids)
    return n / st.median(ts), o.text

base = LLM(model=MODEL, max_model_len=MAX_LEN, gpu_memory_utilization=0.6)
base_tps, base_txt = time_single(base); shutdown(base); del base

spec = LLM(model=MODEL, max_model_len=MAX_LEN, gpu_memory_utilization=0.6,
           speculative_config={"method": "ngram", "num_speculative_tokens": 4,
                               "prompt_lookup_min": 2, "prompt_lookup_max": 4})
spec_tps, spec_txt = time_single(spec); shutdown(spec); del spec

print(f"baseline   : {base_tps:6.0f} tok/s\nn-gram spec: {spec_tps:6.0f} tok/s  →  {spec_tps/base_tps:.2f}× speedup")
print("identical output (greedy):", base_txt == spec_txt)

**Try it:** replace the code-editing prompt with an open-ended one ("write a poem about GPUs"). Nothing in the prompt can be copied, the acceptance rate collapses, and the speedup disappears. Spec decode is **workload-dependent**.

| Method | Drafter | When |
|---|---|---|
| `ngram` | copy from prompt | edits, RAG, summarization |
| `draft_model` | small model, same tokenizer | general |
| `eagle3` | tiny head on target's hidden states | best general acceptance; needs a trained head |
| `mtp` | multi-token-prediction layers built into the model | models trained with MTP (e.g. DeepSeek-V3) |

> **Watch out.** Spec decode shines at **low concurrency**. Under heavy load the GPU is already busy and gains shrink or turn negative. Benchmark at your real traffic level.

## 13. Structured output on the server (JSON schema)

vLLM can **guarantee** schema-valid JSON by masking invalid tokens at every decode step (grammar-constrained decoding). Use it for extraction, tool calls and agents.

In [ ]:
# This section talks to a running vLLM server; start one if it isn't up
if not (isinstance(globals().get("server"), subprocess.Popen) and server.poll() is None):
    server = vllm_serve()

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "gpu": {"type": "string"},
        "memory_gb": {"type": "integer"},
        "bandwidth_tb_s": {"type": "number"},
        "good_for": {"type": "array", "items": {"type": "string"}, "maxItems": 3},
    },
    "required": ["gpu", "memory_gb", "bandwidth_tb_s", "good_for"],
}
resp = client.chat.completions.create(
    model=MODEL, max_tokens=200,
    messages=[{"role": "user", "content": "Describe the NVIDIA H100 SXM GPU for LLM inference."}],
    response_format={"type": "json_schema", "json_schema": {"name": "gpu_card", "schema": schema}})
print(json.dumps(json.loads(resp.choices[0].message.content), indent=2))

> **Watch out.** The **format** is guaranteed, the **facts** are not. A 1.5B model may get specs wrong. Constrained decoding fixes shape, not knowledge.

In [ ]:
stop(server); print("server stopped |", gpu_mem())

## 14. Scoring text instead of generating it: prompt log-probabilities

`prompt_logprobs=0` returns the log-probability of every **prompt** token under the model, with `max_tokens=1` so almost no decode happens. That's a prefill-only workload, which vLLM batches extremely well. Data scientists use it for:

- **Perplexity filtering** of training or synthetic data (drop the weirdest 5%)
- **Comparing answers** by likelihood (which of these completions does the model find most natural?)
- **Checking a quantized model** against the original (§17 uses exactly this)

In [ ]:
# The next sections share one offline engine
if "llm" not in globals():
    llm = LLM(model=MODEL, max_model_len=MAX_LEN, gpu_memory_utilization=0.6, seed=0)

In [ ]:
def perplexity(llm, texts):
    sp = SamplingParams(max_tokens=1, prompt_logprobs=0)
    outs = llm.generate(texts, sp, use_tqdm=False)
    res = []
    for o in outs:
        lps = [next(iter(d.values())).logprob for d in o.prompt_logprobs[1:]]   # first token has no context
        res.append(math.exp(-sum(lps) / len(lps)))
    return res

import math
samples = ["The KV cache stores keys and values for tokens already processed.",
           "Decode is usually limited by memory bandwidth rather than compute.",
           "Bandwidth cache the of tokens decode keys stores usually.",
           "asdkj qwe zzkx plorf vvnm"]
t0 = time.perf_counter()
for s, p in zip(samples, perplexity(llm, samples)):
    print(f"{p:10.1f}  {s}")
print(f"scored in {time.perf_counter() - t0:.2f}s; try it on 10,000 rows of your own data")

## 15. Beam search

Beam search keeps the `beam_width` most likely partial sequences at every step. It's rarely what you want for chat (it produces bland text), but it's useful for short, high-precision outputs such as translation, entity normalisation or query rewriting. In vLLM it's a separate method with its own parameter class.

In [ ]:
from vllm.sampling_params import BeamSearchParams
outs = llm.beam_search([{"prompt": "The full name of the company that makes the H100 GPU is"}],
                       BeamSearchParams(beam_width=4, max_tokens=12))
for seq in outs[0].sequences:
    print(f"{seq.cum_logprob:8.2f}  {seq.text!r}")

## 16. Structured outputs: five kinds of constraint

`StructuredOutputsParams` masks invalid tokens at every decode step, so the output **always** matches. One of these fields per request:

| Field | Use it for |
|---|---|
| `json=` (JSON Schema or a Pydantic model's schema) | Extraction, tool arguments, anything a program parses |
| `regex=` | IDs, dates, phone numbers, fixed formats |
| `choice=[...]` | Classification into a fixed label set (§19) |
| `grammar=` (EBNF) | Small DSLs: SQL subsets, filter expressions |
| `json_object=True` | "Any valid JSON", no schema |

On the server the same object goes in `extra_body={"structured_outputs": {...}}`; `response_format={"type": "json_schema", ...}` also works for JSON (§13). The backend (xgrammar by default, guidance as an alternative) is chosen with `--structured-outputs-config`.

In [ ]:
from pydantic import BaseModel
from vllm.sampling_params import StructuredOutputsParams

class GPU(BaseModel):
    name: str
    memory_gb: int
    hbm_bandwidth_tb_s: float

ask = lambda q: [{"role": "user", "content": q}]
cases = {
    "json": StructuredOutputsParams(json=GPU.model_json_schema()),
    "regex": StructuredOutputsParams(regex=r"[A-Z][0-9]{3} (SXM|PCIe)"),
    "choice": StructuredOutputsParams(choice=["compute-bound", "bandwidth-bound", "overhead-bound"]),
    "grammar": StructuredOutputsParams(grammar=r'''
root   ::= "SELECT " cols " FROM gpus WHERE " cond
cols   ::= col (", " col)*
col    ::= "name" | "memory_gb" | "bandwidth"
cond   ::= col " > " [0-9]+
'''),
}
prompts = {"json": "Describe the H100 SXM as JSON.", "regex": "Name one NVIDIA datacenter GPU and its form factor.",
           "choice": "Batch-1 decode of a 70B model on one H100 is:",
           "grammar": "Write SQL to find GPUs with more than 100 GB of memory."}
for k, so in cases.items():
    o = llm.chat(ask(prompts[k]), SamplingParams(temperature=0, max_tokens=120, structured_outputs=so), use_tqdm=False)
    print(f"{k:8} {o[0].outputs[0].text}")
print(GPU.model_validate_json(llm.chat(ask(prompts["json"]), SamplingParams(temperature=0, max_tokens=120, structured_outputs=cases["json"]), use_tqdm=False)[0].outputs[0].text))

In [ ]:
shutdown(llm); del llm

## 17. Quantize your own checkpoint and check what it cost

[`llm-compressor`](https://github.com/vllm-project/llm-compressor) is the vLLM project's quantization library. Its output (compressed-tensors format) loads in vLLM with no flags. FP8 with dynamic activation scales needs no calibration data and is the safest first step on Hopper, Ada or Blackwell.

Then measure the quality cost with §14's perplexity on text that looks like your traffic. A perplexity increase under ~1–2% is typical for FP8; anything bigger deserves a task-level eval before you ship it.

In [ ]:
%pip install -q llmcompressor
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto")
tokz = AutoTokenizer.from_pretrained(MODEL)
oneshot(model=model, recipe=QuantizationModifier(targets="Linear", scheme="FP8_DYNAMIC", ignore=["lm_head"]))
FP8_DIR = MODEL.split("/")[-1] + "-FP8-dynamic"
model.save_pretrained(FP8_DIR); tokz.save_pretrained(FP8_DIR)
del model

In [ ]:
eval_texts = [f"In LLM serving, {t}" for t in [
    "the KV cache grows linearly with context length and batch size.",
    "prefill is compute-bound while decode is limited by memory bandwidth.",
    "quantizing weights to FP8 halves the bytes read per decode step.",
    "continuous batching keeps the GPU busy by refilling finished slots.",
    "tensor parallelism splits every layer across GPUs and needs fast interconnect."]] * 20

ppl = {}
for name, path in [("bf16", MODEL), ("fp8", FP8_DIR)]:
    m = LLM(model=path, max_model_len=1024, gpu_memory_utilization=0.5)
    ppl[name] = float(np.mean(perplexity(m, eval_texts)))
    shutdown(m); del m
print({k: round(v, 3) for k, v in ppl.items()}, f"→ FP8 changes perplexity by {100 * (ppl['fp8'] / ppl['bf16'] - 1):+.2f}%")

## 18. Pooling models: embeddings, reranking and classification

The same engine (continuous batching, paged memory, CUDA graphs) runs encoder-style models with `runner="pooling"`. There's no decode phase, so these jobs are pure prefill and are compute-bound: big batches, high throughput.

| Offline | Server endpoint | Model type |
|---|---|---|
| `llm.embed(texts)` | `POST /v1/embeddings` (OpenAI format) | Embedding models (bge, e5, gte, Qwen3-Embedding) |
| `llm.score(query, docs)` | `POST /v1/score`, `POST /v1/rerank` (Jina/Cohere format) | Cross-encoders / rerankers |
| `llm.classify(texts)` | `POST /classify` | Sequence-classification heads |

A retrieval pipeline is typically embed → top-50 by cosine → rerank → top-5. Both stages run on one GPU here.

In [ ]:
import numpy as np
docs = ["PagedAttention stores the KV cache in fixed-size blocks.",
        "Speculative decoding drafts tokens with a small model and verifies them in one pass.",
        "FP8 halves weight memory and roughly doubles tensor-core throughput on Hopper.",
        "Continuous batching admits new requests as soon as a slot frees up.",
        "The H200 has 141 GB of HBM3e at 4.8 TB/s."]
query = "How does vLLM avoid wasting KV cache memory?"

emb = LLM(model="BAAI/bge-small-en-v1.5", runner="pooling", gpu_memory_utilization=0.2)
v = np.array([o.outputs.embedding for o in emb.embed([query] + docs, use_tqdm=False)])
v /= np.linalg.norm(v, axis=1, keepdims=True)
cos = v[1:] @ v[0]
shutdown(emb); del emb

rr = LLM(model="BAAI/bge-reranker-v2-m3", runner="pooling", gpu_memory_utilization=0.3)
scores = [o.outputs.score for o in rr.score(query, docs, use_tqdm=False)]
shutdown(rr); del rr

display(pd.DataFrame({"doc": [d[:60] for d in docs], "cosine": cos.round(3), "rerank": np.round(scores, 3)})
        .sort_values("rerank", ascending=False))

## 19. The data-scientist workloads: embeddings and zero-shot labelling at scale

Two jobs that show up in every DS team and run far faster through an inference engine than through a `for` loop over `transformers`.

In [ ]:
from vllm import LLM, SamplingParams

# 1) Embeddings for 2,000 texts with a pooling model
texts = [f"Customer {i} says the delivery was {'late' if i % 3 else 'on time'} and the item was {'damaged' if i % 5 == 0 else 'fine'}." for i in range(2000)]
try:
    emb_llm = LLM(model="BAAI/bge-small-en-v1.5", runner="pooling", gpu_memory_utilization=0.3)
except TypeError:                                    # older vLLM used task="embed"
    emb_llm = LLM(model="BAAI/bge-small-en-v1.5", task="embed", gpu_memory_utilization=0.3)
t0 = time.perf_counter(); out = emb_llm.embed(texts); dt_ = time.perf_counter() - t0
print(f"{len(texts)} embeddings of dim {len(out[0].outputs.embedding)} in {dt_:.1f}s → {len(texts)/dt_:.0f} texts/s")
shutdown(emb_llm)

In [ ]:
# 2) Zero-shot labelling with a constrained label set, plus a confidence score from logprobs
llm = LLM(model=MODEL, max_model_len=1024, gpu_memory_utilization=0.6)
LABELS = ["late_delivery", "damaged_item", "positive"]
try:
    from vllm.sampling_params import StructuredOutputsParams
    sp = SamplingParams(temperature=0, max_tokens=8, logprobs=1, structured_outputs=StructuredOutputsParams(choice=LABELS))
except ImportError:                                  # older vLLM name
    from vllm.sampling_params import GuidedDecodingParams
    sp = SamplingParams(temperature=0, max_tokens=8, logprobs=1, guided_decoding=GuidedDecodingParams(choice=LABELS))
msgs = [[{"role": "user", "content": f"Label this review as one of {LABELS}.\nReview: {t}"}] for t in texts[:1000]]
t0 = time.perf_counter(); outs = llm.chat(msgs, sp); dt_ = time.perf_counter() - t0
import math
rows = [{"text": t[:60], "label": o.outputs[0].text,
         "confidence": math.exp(sum(next(iter(lp.values())).logprob for lp in o.outputs[0].logprobs))} for t, o in zip(texts, outs)]
display(pd.DataFrame(rows).head(8))
print(f"1,000 labels in {dt_:.1f}s. Every output is guaranteed to be one of {LABELS}.")
shutdown(llm)

> Constrained choices remove the parsing step and the "model answered in a sentence" failure mode. The logprob-based confidence lets you route low-confidence rows to a bigger model or a human: a cheap way to cut labelling cost.

## 20. Multi-LoRA serving: the server form

Serve many fine-tuned adapters on **one** copy of the base model. Each request picks its adapter; requests for different adapters are batched together.

```bash
# server form
vllm serve Qwen/Qwen2.5-1.5B-Instruct --enable-lora \
    --lora-modules support=/path/to/support_adapter sql=/path/to/sql_adapter
# clients just set model="support" or model="sql"
```

Set `LORA_PATH` to a local or Hugging Face LoRA adapter trained on this base model to run the offline version:

In [ ]:
LORA_PATH = None   # e.g. a HF repo id or local folder of a LoRA adapter for Qwen2.5-1.5B-Instruct

if LORA_PATH:
    from vllm.lora.request import LoRARequest
    from huggingface_hub import snapshot_download
    path = LORA_PATH if os.path.isdir(LORA_PATH) else snapshot_download(LORA_PATH)
    llm = LLM(model=MODEL, enable_lora=True, max_lora_rank=64, max_model_len=MAX_LEN, gpu_memory_utilization=0.6)
    sp = SamplingParams(temperature=0, max_tokens=80)
    q = [{"role": "user", "content": "Introduce yourself."}]
    print("base   :", llm.chat(q, sp)[0].outputs[0].text)
    print("adapter:", llm.chat(q, sp, lora_request=LoRARequest("my_adapter", 1, path))[0].outputs[0].text)
    shutdown(llm); del llm
else:
    print("Set LORA_PATH to try multi-LoRA serving.")

## 21. LoRA adapters: make one, serve several, hot-load at runtime

To show the mechanics without depending on someone else's adapter, we create a small random LoRA with `peft` (it changes the output, which is all we need to see routing work). With a real fine-tune the code is identical.

Key engine arguments: `enable_lora=True`, `max_loras` (adapters active in **one batch**), `max_lora_rank` (largest rank you'll load), `max_cpu_loras` (adapters cached in CPU RAM). Different adapters in the same batch run together through Punica-style batched kernels, so ten fine-tunes cost about one base model's memory.

In [ ]:
%pip install -q peft
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

ADAPTERS = {}
for name, seed in [("adapter_a", 1), ("adapter_b", 2)]:
    torch.manual_seed(seed)
    base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16)
    m = get_peft_model(base, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], init_lora_weights=False))
    m.save_pretrained(f"./{name}"); ADAPTERS[name] = os.path.abspath(f"./{name}")
    del base, m
print(ADAPTERS)

In [ ]:
from vllm.lora.request import LoRARequest
llm = LLM(model=MODEL, enable_lora=True, max_loras=2, max_lora_rank=16, max_model_len=1024, gpu_memory_utilization=0.6)
sp = SamplingParams(temperature=0, max_tokens=24)
q = [{"role": "user", "content": "Say hello in five words."}]
requests = [None, LoRARequest("adapter_a", 1, ADAPTERS["adapter_a"]), LoRARequest("adapter_b", 2, ADAPTERS["adapter_b"])]
for r in requests:                                     # in production these run in one batch
    print(f"{(r.lora_name if r else 'base'):10}", llm.chat(q, sp, lora_request=r, use_tqdm=False)[0].outputs[0].text.strip()[:80])
shutdown(llm); del llm

On the server, adapters can be listed at start-up (`--lora-modules name=path`) or loaded while it runs. Runtime loading needs `VLLM_ALLOW_RUNTIME_LORA_UPDATING=True`; it's off by default because anyone who can reach the endpoint can load weights.

In [ ]:
server = vllm_serve("--enable-lora", "--max-lora-rank", "16", "--lora-modules", f"adapter_a={ADAPTERS['adapter_a']}",
                    env={"VLLM_ALLOW_RUNTIME_LORA_UPDATING": "True"})
import requests
print([m["id"] for m in requests.get(f"{BASE}/v1/models").json()["data"]])
requests.post(f"{BASE}/v1/load_lora_adapter", json={"lora_name": "adapter_b", "lora_path": ADAPTERS["adapter_b"]}).raise_for_status()
print([m["id"] for m in requests.get(f"{BASE}/v1/models").json()["data"]])
print(measure_stream(q, model="adapter_b", max_tokens=24)["text"])
requests.post(f"{BASE}/v1/unload_lora_adapter", json={"lora_name": "adapter_b"}).raise_for_status()
stop(server)

## 22. `vllm bench serve`: the official load generator

For real capacity planning, use the built-in benchmark with **your** prompt/output length distribution. Here: random 512-token prompts, 128-token outputs, 32 concurrent users.

In [ ]:
# This section talks to a running vLLM server; start one if it isn't up
if not (isinstance(globals().get("server"), subprocess.Popen) and server.poll() is None):
    server = vllm_serve()

In [ ]:
!vllm bench serve --model {MODEL} --port {PORT} \
    --dataset-name random --random-input-len 512 --random-output-len 128 \
    --num-prompts 200 --max-concurrency 32 2>&1 | tail -n 30

**Try it:** Re-run with `--max-concurrency 1, 8, 64` and with `--random-input-len 4000`. Which metric moves most when prompts get long? (Hint: prefill → TTFT.)

**Stop the server before the next section:**

In [ ]:
stop(server); print("server stopped |", gpu_mem())

SGLang (LMSYS) implements the same toolkit as vLLM (continuous batching, paged KV, chunked prefill, CUDA graphs, spec decode, TP/DP/EP, PD disaggregation). Where it differs:

| | vLLM | SGLang |
|---|---|---|
| Prefix reuse | Hash-based automatic prefix caching (block level) | **RadixAttention**: a radix tree over all cached tokens, with LRU eviction and a prefix-aware scheduler (`--schedule-policy lpm`) |
| Programming model | OpenAI API + offline `LLM` | OpenAI API + offline `sgl.Engine` + a **frontend language** (`fork`, `gen`, `select`) for multi-call programs |
| Big MoE | EP, DP attention | EP, **DP attention** (`--enable-dp-attention`), DeepEP; many DeepSeek/Kimi day-0 recipes come from here |
| Defaults to know | `--gpu-memory-utilization 0.9` | `--mem-fraction-static` (default depends on GPU) |

Modal's LLM Engine Almanac found the two **perform comparably out of the box**. Choose on features, model support and your traffic shape, and benchmark with your own prompts (§27 does exactly that).

> **Watch out.** **Environment edge case.** vLLM, SGLang and TensorRT-LLM each pin their own PyTorch/CUDA versions. Installing all three in one environment usually breaks at least one of them. Use **one virtualenv (or container) per engine**. The cells below launch each engine as a *subprocess*, so this notebook's kernel only needs `openai`, `requests` and `pandas`. Point `SGLANG_PY` at the Python of your SGLang env.

## 23. Install & launch an SGLang server

```bash
# separate env, e.g.
uv venv ~/.venvs/sglang && source ~/.venvs/sglang/bin/activate
uv pip install "sglang[all]"          # check docs.sglang.ai for the CUDA-specific wheel if this fails
```

In [ ]:
import os, sys, subprocess, time, urllib.request, json
import requests

# Free the GPU from earlier parts: two engines on one GPU will fight over memory
try:
    if server.poll() is None: server.terminate(); server.wait(60)
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

SGLANG_PY = os.environ.get("SGLANG_PY", sys.executable)   # e.g. ~/.venvs/sglang/bin/python
SG_PORT = 30000
SG_BASE = f"http://localhost:{SG_PORT}"

def launch(cmd, logfile, health_url, timeout=900):
    # Start an engine as a subprocess and block until /health answers.
    log = open(logfile, "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            raise RuntimeError(f"engine exited; tail {logfile}")
        try:
            if urllib.request.urlopen(health_url, timeout=2).status == 200:
                print(f"ready in {time.time() - t0:.0f}s"); return proc
        except Exception:
            time.sleep(2)
    proc.terminate(); raise TimeoutError(f"not healthy; see {logfile}")

sg = launch([SGLANG_PY, "-m", "sglang.launch_server",
             "--model-path", MODEL, "--port", str(SG_PORT),
             "--context-length", str(MAX_LEN),
             "--mem-fraction-static", "0.80",
             "--enable-metrics"],
            "sglang_server.log", f"{SG_BASE}/health")

In [ ]:
# The capacity lines SGLang prints at startup (wording varies a little by version)
for line in open("sglang_server.log"):
    if any(k in line for k in ["max_total_num_tokens", "KV Cache is allocated", "max_running_requests", "Load weight end"]):
        print(line.strip()[-160:])

> `max_total_num_tokens` is SGLang's equivalent of vLLM's "KV cache size in tokens". Divide by your typical context length to get a rough max concurrency, same arithmetic as Session 1 §4.

## 24. Same OpenAI API, plus SGLang's native `/generate`

The OpenAI-compatible endpoint means **everything from Part B works unchanged**: just swap the base URL. The native `/generate` endpoint adds `meta_info`, which reports `cached_tokens`: how many prompt tokens were served from the radix cache.

In [ ]:
from openai import OpenAI
sg_client = OpenAI(base_url=f"{SG_BASE}/v1", api_key="none")
r = sg_client.chat.completions.create(model=MODEL, max_tokens=60,
        messages=[{"role": "user", "content": "One sentence: what is RadixAttention?"}])
print(r.choices[0].message.content)

import requests
def sg_generate(text, max_new_tokens=32):
    out = requests.post(f"{SG_BASE}/generate", json={
        "text": text, "sampling_params": {"max_new_tokens": max_new_tokens, "temperature": 0}}).json()
    return out["text"], out["meta_info"]

_, meta = sg_generate("Hello! " * 5)
print({k: meta[k] for k in ("prompt_tokens", "completion_tokens", "cached_tokens") if k in meta})

## 25. RadixAttention: watch `cached_tokens` and TTFT

Same experiment as §3, but now the engine *tells* us how much it reused. We build a ~3k-token shared prefix (think: a long system prompt or a RAG document), then send different questions behind it.

In [ ]:
DOC = " ".join(f"Clause {i}: the supplier shall deliver item {i} within {i % 30 + 1} days." for i in range(220))
QUESTIONS = ["Which clause covers item 17?", "Summarise clause 3.", "What is the longest delivery time?",
             "List clauses with 5-day delivery.", "Is item 200 covered?"]

def timed(text):
    t0 = time.perf_counter(); _, m = sg_generate(text, max_new_tokens=1)   # 1 token ≈ pure prefill time
    return (time.perf_counter() - t0) * 1e3, m.get("cached_tokens", 0), m["prompt_tokens"]

requests.post(f"{SG_BASE}/flush_cache")           # start cold
rows = []
for i, q in enumerate(QUESTIONS):
    ms, cached, total = timed(f"{DOC}\n\nQuestion: {q}\nAnswer:")
    rows.append({"request": i, "prompt_tokens": total, "cached_tokens": cached, "ttft_ms": round(ms, 1)})
# Control: same length, but the dynamic part is FIRST, so nothing can be reused
ms, cached, total = timed(f"Request id {time.time()}.\n{DOC}\n\nQuestion: {QUESTIONS[0]}\nAnswer:")
rows.append({"request": "dynamic-first", "prompt_tokens": total, "cached_tokens": cached, "ttft_ms": round(ms, 1)})
display(pd.DataFrame(rows))

> Request 0 pays for the whole prefix; requests 1–4 hit the radix tree and only prefill the question. The `dynamic-first` row is the classic mistake: a timestamp or request ID at the top of the prompt makes the cache useless. **Static first, dynamic last.**

## 26. The SGLang frontend: `fork` for parallel sub-calls

For multi-step LLM programs (judge with several criteria, tree-of-thought, agents), SGLang's frontend lets you write the program once. `fork` branches share the prefix's KV cache automatically, so you pay for the shared context once.

In [ ]:
try:
    import sglang as sgl
    sgl.set_default_backend(sgl.RuntimeEndpoint(SG_BASE))

    @sgl.function
    def judge(s, answer):
        s += sgl.system("You are a strict evaluator. Reply with a score 1-5 and one short reason.")
        s += sgl.user(f"Answer to evaluate:\n{answer}")
        forks = s.fork(3)
        for f, crit in zip(forks, ["factual accuracy", "clarity", "conciseness"]):
            f += sgl.user(f"Score it on {crit}.")
            f += sgl.assistant(sgl.gen("score", max_tokens=40, temperature=0))
        s["scores"] = [f["score"] for f in forks]

    st_ = judge.run(answer="PagedAttention stores the KV cache in fixed-size blocks, like OS virtual memory pages.")
    for c, sc in zip(["accuracy", "clarity", "concise"], st_["scores"]): print(f"{c:9} → {sc.strip()}")
except ImportError:
    print("The frontend needs `sglang` importable in THIS kernel. Skip, or run this cell from the SGLang env.")

> For data scientists this is the LLM-as-judge pattern: one shared (expensive) context, several cheap criteria. Same idea works in vLLM via prefix caching, but here it's explicit.

### SGLang flags worth knowing

| Flag | What it does | vLLM equivalent |
|---|---|---|
| `--tp N` / `--dp N` | tensor / data parallel | `--tensor-parallel-size` / `--data-parallel-size` |
| `--mem-fraction-static 0.85` | share of VRAM for weights + KV pool | `--gpu-memory-utilization` |
| `--context-length` | max context | `--max-model-len` |
| `--max-running-requests` | concurrency cap | `--max-num-seqs` |
| `--chunked-prefill-size 4096` | prefill chunk per step | `--max-num-batched-tokens` |
| `--quantization fp8`, `--kv-cache-dtype fp8_e5m2` | weights / KV precision | same names |
| `--speculative-algorithm EAGLE3 --speculative-draft-model-path …` | EAGLE spec decoding | `--speculative-config` |
| `--enable-dp-attention` | DP for attention + EP for experts (DeepSeek-style MoE) | `--data-parallel-size` + `--enable-expert-parallel` |
| `--disable-radix-cache` | turn prefix reuse off (A/B tests) | `--no-enable-prefix-caching` |
| `--disaggregation-mode prefill\|decode` | PD disaggregation | KV connector config |

Always confirm with `python -m sglang.launch_server --help` for your version; flags move between releases.

## 27. One harness, any engine

Every engine here speaks the OpenAI API, so a single async load generator gives an apples-to-apples comparison. We reuse it for TensorRT-LLM in Part E.

In [ ]:
import asyncio
from openai import AsyncOpenAI

async def bench_openai(base, model, levels=(1, 8, 32, 64), max_tokens=256, prompt_words=200):
    ac = AsyncOpenAI(base_url=f"{base}/v1", api_key="none")
    filler = " ".join(["data"] * prompt_words)
    async def one(i):
        t0 = time.perf_counter(); first = last = None; n = 0
        stream = await ac.chat.completions.create(model=model, max_tokens=max_tokens, stream=True,
            stream_options={"include_usage": True}, extra_body={"ignore_eos": True},
            messages=[{"role": "user", "content": f"[{i}] {filler}\nWrite a long story about GPU {i}."}])
        async for c in stream:
            if c.choices and c.choices[0].delta.content:
                now = time.perf_counter(); first = first or now; last = now
            if c.usage: n = c.usage.completion_tokens
        return first - t0, (last - first) / max(n - 1, 1), n
    rows = []
    for c in levels:
        await asyncio.gather(*[one(i) for i in range(min(c, 4))])          # warm-up
        t0 = time.perf_counter(); res = await asyncio.gather(*[one(i) for i in range(c)])
        wall = time.perf_counter() - t0; ttft, tpot, ns = zip(*res)
        rows.append({"concurrency": c, "ttft_p50_ms": st.median(ttft) * 1e3, "tpot_p50_ms": st.median(tpot) * 1e3,
                     "per_user_tok_s": 1 / st.median(tpot), "system_tok_s": sum(ns) / wall})
    return pd.DataFrame(rows).round(1)

results = {}
results["sglang"] = await bench_openai(SG_BASE, MODEL)
display(results["sglang"])

In [ ]:
# Official load generator. It can also target vLLM / TRT-LLM servers via --backend / --base-url.
!{SGLANG_PY} -m sglang.bench_serving --backend sglang --port {SG_PORT} \
    --dataset-name random --random-input-len 512 --random-output-len 256 \
    --num-prompts 200 --max-concurrency 32 2>&1 | tail -25

In [ ]:
sg.terminate(); sg.wait(60); print("SGLang stopped")

## 28. Tuning playbook & cheat sheet *(reading)*

### Symptom → lever

| Symptom | Try (roughly in order) |
|---|---|
| **TTFT too high** | prefix caching + static-first prompts → shorter prompts → tune `max_num_batched_tokens` → more replicas / TP → disaggregate prefill |
| **TPOT too slow** | FP8 / INT4 weights → speculative decoding → TP → lower `max_num_seqs` → GPU with more bandwidth |
| **Throughput too low** | raise `max_num_seqs` → more KV room (`gpu_memory_utilization`, FP8 KV) → quantize → DP replicas |
| **OOM / preemptions** | lower `max_model_len` → FP8 KV cache → fewer concurrent seqs → TP / bigger GPU |
| **Cost too high** | smaller or distilled model → quantize → right-size GPU → batch offline jobs → raise utilization |

### The method
1. **Define the SLO** (e.g. TTFT P99 < 1 s, ≥ 30 tok/s per user).
2. **Load-test realistic traffic**: real prompt and output length distributions, real concurrency.
3. **Find the bottleneck**: prefill (TTFT) or decode (TPOT)? Memory (KV usage, preemptions) or compute?
4. **Change one knob**, re-measure, keep what helps.

### Cost formula
$$\text{\$ per 1M tokens} = \frac{\text{GPU \$/hour}}{\text{tokens/s} \times 3600} \times 10^6$$

In [ ]:
def cost_per_million(gpu_dollars_per_hour, tokens_per_second):
    return gpu_dollars_per_hour / (tokens_per_second * 3600) * 1e6

# Illustrative prices; plug in your own GPU price and the throughput you measured in Session 1 §15 or §22 here
for price, tps in [(3.0, 500), (3.0, 2000), (3.0, 8000)]:
    print(f"${price}/hr GPU at {tps:>5} tok/s  →  ${cost_per_million(price, tps):.2f} per 1M output tokens")

> Same GPU, same price: **utilization is the cost lever**. Batching, prefix caching and quantization all show up here as a smaller number.

---

## Exercises

1. **Tune the scheduler.** In §1, change `SLOTS` and the output-length mix. When does continuous batching's advantage over static batching shrink?
2. **Break prefix caching.** Add `f"Current time: {time.time()}"` to the *top* of the system prompt in §3. What happens to warm TTFT? Now move it to the bottom.
3. **Quantize harder.** In the quantization section, try group sizes 32, 64 and 256 for INT4. Plot perplexity against bits per weight including the scales.
4. **Make spec decode lose.** Raise concurrency with n-gram speculation on. At what concurrency does it stop helping on your GPU?
5. **Your own data.** Use prompt-logprob scoring to rank 1,000 rows of a dataset you own by perplexity and read the bottom 20.

In [ ]:
# Stop anything this notebook started
for _name in ("server", "sg", "trt"):
    if isinstance(globals().get(_name), subprocess.Popen):
        stop(globals()[_name])
print(gpu_mem())

Next: **Session 3, Advanced**: connecting GPUs, every kind of parallelism, big models on 8 GPUs, disaggregation and production. [`Session3_Advanced.ipynb`](Session3_Advanced.ipynb)